In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import geopandas as gpd
import geobr
import polars as pl
import pandas as pd
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 50)
from scripts.configs import DIRS
from scripts.utils import ler_arquivo_polars
from shapely.geometry import Point

In [3]:
df = pd.read_parquet('dados/tabela_final.parquet')
df = df.fillna(0)
df['uf_nome'] = df['uf'] + ' - ' + df['nome']
df['obitos_fora_estabelecimento'] = df['total_obitos'] - df['obitos_em_estabelecimentos']
df['taxa_obitos_fora_estabelecimento'] = (df['obitos_fora_estabelecimento'] / df['total_obitos']) * 100

In [ ]:
cnes = pd.read_csv('data/raw/tbEstabelecimento202212.csv', sep=';', encoding='latin1', low_memory=False)
cnes = cnes[cnes['CO_MOTIVO_DESAB'].isnull()]

In [ ]:
cnes[
    (cnes['CO_MUNICIPIO_GESTOR'] == 411820) &
    (cnes['NO_FANTASIA'].str.contains('regional', case=False))
]

In [ ]:
cnes['NU_LONGITUDE'] = cnes['NU_LONGITUDE'].str.replace(',', '.').astype(float)
cnes['NU_LATITUDE'] = cnes['NU_LATITUDE'].str.replace(',', '.').astype(float)

In [ ]:
coordenadas = [Point(xy) for xy in zip(cnes['NU_LONGITUDE'], cnes['NU_LATITUDE'])]

In [ ]:
cnes_geo = gpd.GeoDataFrame(cnes, geometry=coordenadas, crs='EPSG:4326')

ax = geobr.read_state().plot(
    figsize=(12, 12),
    color='white',
    edgecolor='gray'
)

cnes_geo.plot(
    ax=ax,
    color='blue',
    markersize=5,
    alpha=0.5
)

plt.title('Distribuição de Estabelecimentos de Saúde no Brasil', fontsize=16)
plt.axis('off')
plt.show()

In [ ]:
# Jeito de plotar o mapa mais rudimentar

import plotly.express as px

# Converter o GeoDataFrame para DataFrame comum com colunas lat/lon
cnes_plot = cnes_geo.copy()
cnes_plot['lat'] = cnes_plot.geometry.y
cnes_plot['lon'] = cnes_plot.geometry.x

# Criar o mapa interativo com as colunas que realmente existem no DataFrame
fig = px.scatter_map(
    cnes_plot,
    lat='lat', 
    lon='lon',
    hover_name='NO_FANTASIA',  # Nome que aparece em destaque
    hover_data={  # Dados extras que aparecem ao passar o mouse
        'CO_CNES': True,
        'TP_UNIDADE': True,
        # Remova as colunas que não existem
        # 'NO_MUNICIPIO': True,  
        # 'CO_SIGLA_ESTADO': True,
        'CO_MUNICIPIO_GESTOR': True,  # Use esta coluna em vez de NO_MUNICIPIO
        'CO_ESTADO_GESTOR': True,     # Use esta coluna em vez de CO_SIGLA_ESTADO
        'lat': False,  # Esconde lat/lon do hover
        'lon': False
    },
    zoom=4,
    height=800,
    width=1000,
    title='Distribuição de Estabelecimentos de Saúde no Brasil',
    mapbox_style='carto-positron'  # ou 'open-street-map', 'white-bg', etc.
)

# Melhorar o layout
fig.update_layout(
    margin={"r": 0, "t": 30, "l": 0, "b": 0},
)

# Exibir o mapa
fig.show()

In [ ]:
cnes_plot = cnes_geo.copy()
cnes_plot['lat'] = cnes_plot.geometry.y
cnes_plot['lon'] = cnes_plot.geometry.x

# Criar o mapa interativo com as colunas que realmente existem no DataFrame
fig = px.scatter_map(
    cnes_plot,
    lat='lat', 
    lon='lon',
    hover_name='NO_FANTASIA',
    hover_data={
        'CO_CNES': True,
        'TP_UNIDADE': True,
        'CO_MUNICIPIO_GESTOR': True,
        'CO_ESTADO_GESTOR': True,
        'lat': False,
        'lon': False
    },
    zoom=4,
    height=800,
    width=1000
)

# Definir estilo do mapa e título
fig.update_layout(
    mapbox_style='open-street-map',  # ou 'carto-positron', 'stamen-terrain', etc.
    title='Distribuição de Estabelecimentos de Saúde no Brasil',
    margin={"r": 0, "t": 30, "l": 0, "b": 0}
)

fig.show()

In [4]:
municipios = geobr.read_municipality()
municipios['code_muni_abrev'] = municipios['code_muni'].astype(str).str.slice(0, 6)

In [5]:
municipios[municipios['code_muni_abrev'] == '411820']

,code_muni,name_muni,code_state,abbrev_state,geometry,code_muni_abrev
4163,4118204.0,Paranaguá,41.0,PR,"MULTIPOLYGON (((-48.6243 -25.45689, -48.62638 ...",411820


In [ ]:
# Fazer o merge com seus dados
df['codigo_municipio'] = df['codigo_municipio'].astype(str)
municipios_com_dados = municipios.merge(
    df,
    left_on='code_muni_abrev',
    right_on='codigo_municipio',
    how='left'
)


In [ ]:
# Definir a escala de cores (vermelho para verde)
cores = [
    [0, 'rgb(165,0,38)'],    # Vermelho escuro para valores baixos
    [0.25, 'rgb(215,48,39)'], # Vermelho
    [0.5, 'rgb(254,224,144)'], # Amarelo
    [0.75, 'rgb(116,196,118)'], # Verde claro
    [1, 'rgb(35,139,69)']     # Verde escuro para valores altos
]

# Criar o mapa choropleth com personalização
fig = px.choropleth_map(
    municipios_com_dados,
    geojson=municipios_com_dados.geometry,
    locations=municipios_com_dados.index,
    color='unidades_por_k_hab',  # Colorir por unidades por k habitantes
    color_continuous_scale=cores,  # Usar nossa escala personalizada vermelho->verde
    range_color=[0, municipios_com_dados['unidades_por_k_hab'].quantile(0.95)],  # Ajustar a escala para melhor visualização
    zoom=3.5,
    center={"lat": -15.77972, "lon": -47.92972},
    opacity=0.7,
    hover_name='nome',  # Nome do município para exibir no hover
    hover_data=[  # Lista de colunas para mostrar no hover
        'populacao', 
        'idh', 
        'qtd_unidades', 
        'unidades_por_k_hab',
        'medicos_por_k_habitante',
        'enfermeiros_por_k_habitante',
        'leitos_por_k_hab',
        'taxa_mortalidade_geral'
    ],
    labels={  # Renomear rótulos para melhor legibilidade no hover
        'unidades_por_k_hab': 'Unidades por mil hab.',
        'populacao': 'População',
        'idh': 'IDH',
        'qtd_unidades': 'Total de unidades',
        'medicos_por_k_habitante': 'Médicos por mil hab.',
        'enfermeiros_por_k_habitante': 'Enfermeiros por mil hab.',
        'leitos_por_k_hab': 'Leitos por mil hab.',
        'taxa_mortalidade_geral': 'Taxa de mortalidade'
    },
    title='Acesso à Saúde nos Municípios Brasileiros<br><sup>Coloração por Unidades de Saúde por mil habitantes</sup>'
)

# Ajustar layout
fig.update_layout(
    mapbox_style="carto-positron",
    margin={"r":0,"t":50,"l":0,"b":0},
    height=800,
    coloraxis_colorbar=dict(
        title="Unidades por<br>mil hab.",
        thicknessmode="pixels", thickness=20,
        lenmode="pixels", len=300,
        yanchor="top", y=1,
        ticks="outside"
    )
)

# Formatar os valores nas tooltips para dois decimais
fig.update_traces(
    hovertemplate='<b>%{hovertext}</b><br><br>' +
                 'População: %{customdata[0]:,.0f}<br>' +
                 'IDH: %{customdata[1]:.3f}<br>' +
                 'Total de unidades: %{customdata[2]}<br>' +
                 'Unidades por mil hab.: %{customdata[3]:.2f}<br>' +
                 'Médicos por mil hab.: %{customdata[4]:.2f}<br>' +
                 'Enfermeiros por mil hab.: %{customdata[5]:.2f}<br>' +
                 'Leitos por mil hab.: %{customdata[6]:.2f}<br>' +
                 'Taxa de mortalidade: %{customdata[7]:.2f}<br>'
)

fig.show()

In [ ]:
# Definir a escala de cores (vermelho para verde)
cores = [
    [0, 'rgb(165,0,38)'],    # Vermelho escuro para valores baixos
    [0.25, 'rgb(215,48,39)'], # Vermelho
    [0.5, 'rgb(254,224,144)'], # Amarelo
    [0.75, 'rgb(116,196,118)'], # Verde claro
    [1, 'rgb(35,139,69)']     # Verde escuro para valores altos
]

labels = {  # Renomear rótulos para melhor legibilidade no hover
        'unidades_por_k_hab': 'Unidades por mil hab.',
        'populacao': 'População',
        'idh': 'IDH',
        'qtd_unidades': 'Total de unidades',
        'medicos_por_k_habitante': 'Médicos por mil hab.',
        'enfermeiros_por_k_habitante': 'Enfermeiros por mil hab.',
        'quantidade_unidades_com_leito': 'Unidades com leito',
        'leitos_existentes': 'Leitos existentes',
        'leitos_por_k_hab': 'Leitos por mil hab.',
        'leitos_sus': 'Leitos SUS',
        'leitos_sus_por_k_hab': 'Leitos SUS por mil hab.',
        'obitos_por_unidade': 'Óbitos por unidade',
        'obitos_em_estabelecimentos': 'Óbitos em estabelecimentos',
        'obitos_fora_estabelecimento': 'Óbitos fora de estabelecimentos',
        'taxa_obitos_fora_estabelecimento': 'Percentual de óbitos fora de estabelecimentos',
        'total_obitos': 'Total de óbitos',
        'taxa_mortalidade_geral': 'Taxa de mortalidade',
        'mortalidade_infantil': 'Mortalidade infantil'
}

kpi = 'taxa_obitos_fora_estabelecimento'
partes = labels[kpi].split()

# Separar os dois primeiros e o restante
primeiros_dois = ' '.join(partes[:2])
restante = ' '.join(partes[2:])

# Criar o mapa choropleth com personalização
fig = px.choropleth_map(
    municipios_com_dados,
    geojson=municipios_com_dados.geometry,
    locations=municipios_com_dados.index,
    color=kpi,  # Colorir por unidades por k habitantes
    color_continuous_scale=cores,  # Usar nossa escala personalizada vermelho->verde
    range_color=[0, municipios_com_dados[kpi].quantile(0.95)],  # Ajustar a escala para melhor visualização | unidades_por_k_hab
    zoom=3.5,
    center={"lat": -15.77972, "lon": -47.92972},
    opacity=0.7,
    hover_name='uf_nome',  # Nome do município para exibir no hover
    hover_data=[  # Lista de colunas para mostrar no hover
        'populacao', 
        'idh', 
        'qtd_unidades', 
        'unidades_por_k_hab',
        'medicos_por_k_habitante',
        'enfermeiros_por_k_habitante',
        'quantidade_unidades_com_leito',
        'leitos_existentes',
        'leitos_por_k_hab',
        'leitos_sus',
        'leitos_sus_por_k_hab',
        'obitos_por_unidade',
        'obitos_em_estabelecimentos',
        'obitos_fora_estabelecimento',
        'taxa_obitos_fora_estabelecimento',
        'total_obitos',
        'taxa_mortalidade_geral',
        'mortalidade_infantil'
    ],
    labels=labels,
    title='Acesso à Saúde nos Municípios Brasileiros<br><sup>Coloração por Unidades de Saúde por mil habitantes</sup>'
)

# Ajustar layout
fig.update_layout(
    mapbox_style="carto-positron",
    margin={"r":0,"t":50,"l":0,"b":0},
    height=800,
    coloraxis_colorbar=dict(
        title=f"{primeiros_dois}<br>{restante}",
        thicknessmode="pixels", thickness=20,
        lenmode="pixels", len=300,
        yanchor="top", y=1,
        ticks="outside"
    )
)

# Formatar os valores nas tooltips para dois decimais
fig.update_traces(
    hovertemplate='<b>%{hovertext}</b><br><br>' +
                 'População: %{customdata[0]:,.0f}<br>' +
                 'IDH: %{customdata[1]:.3f}<br>' +
                 'Total de unidades: %{customdata[2]}<br>' +
                 'Unidades por mil hab.: %{customdata[3]:.2f}<br>' +
                 'Médicos por mil hab.: %{customdata[4]:.2f}<br>' +
                 'Enfermeiros por mil hab.: %{customdata[5]:.2f}<br>' +
                 'Unidades com leito: %{customdata[6]}<br>' +
                 'Leitos existentes: %{customdata[7]:.0f}<br>' +
                 'Leitos por mil hab.: %{customdata[8]:.2f}<br>' +
                 'Leitos SUS: %{customdata[9]:.0f}<br>' +
                 'Leitos SUS por mil hab.: %{customdata[10]:.2f}<br>' +
                 'Óbitos por unidade: %{customdata[11]:.2f}<br>' +
                 'Óbitos em estabelecimentos: %{customdata[12]:.0f}<br>' +
                 'Óbitos fora de estabelecimentos: %{customdata[13]:.0f}<br>' +
                 'Percentual de óbitos fora de estabelecimentos: %{customdata[14]:.2f}<br>' +
                 'Total de óbitos: %{customdata[15]:.0f}<br>' +
                 'Taxa de mortalidade: %{customdata[16]:.2f}<br>' +
                 'Mortalidade infantil: %{customdata[17]:.2f}<br>'
)

fig.show()